# Smart Logistics IoT Simulation
**Course:** MO-IT148 — Application Development and Emerging Technologies <br>
**Group:** NodeBlk <br><br><br>
**Week:** 2 — IoT Data Simulation <br>
**Description:** Simulates raw sensor data (GPS, RFID, Temperature) for a smart 
logistics tracking system. Generates the Shipment Registry first, then 
attaches 3 independent sensor scripts to it. Only temp-regulated goods 
categories receive Temperature readings. Exports to CSV for blockchain 
processing.<br>

## 1. Setup
Imports and global constants. All counts and bounds live here — generators read from these, never hard-code their own.

### Libraries
| Library | Purpose |
|---------|---------|
| `numpy` | Numerical operations (available for future use) |
| `pandas` | DataFrame creation, manipulation, and CSV export |
| `random` | Random selection of cities, statuses, device IDs, and anomaly generation |
| `datetime` | Timestamp generation and time-interval arithmetic |

### Simulation Constants
| Constant | Value | Description |
|----------|-------|-------------|
| `NUM_SHIPMENTS` | `30` | Total number of shipments to simulate |
| `READINGS_PER_SHIPMENT` | `5` | How many sensor readings each shipment produces per sensor type |
| `START_TIMESTAMP` | `2026-05-03 06:00:00` | The base datetime from which all reading timestamps are calculated |

### Goods Categories
Goods are split into two groups that determine which sensors are attached:

**Temp-Regulated** — receive GPS, RFID, *and* Temperature sensors:
| Category | Safe Temperature Range |
|----------|----------------------|
| `Deep Freeze` | −30°C to −28°C |
| `Frozen` | −20°C to −16°C |
| `Chill/Refrigerated` | 2°C to 4°C |
| `Pharma` | 2°C to 8°C |
| `Cool-Chain` | 12°C to 14°C |

**Non-Temp** — receive GPS and RFID sensors only (no temperature monitoring needed):
`Dry Goods`, `Electronics`, `Clothing`, `Industrial`

### City Data (`ph.csv`)
| Variable | Description |
|----------|-------------|
| `CITIES` | List of Philippine city names used for origin/destination assignment |
| `CITY_COORDS` | Dictionary mapping each city name to its `(lat, lng)` coordinates for GPS interpolation |

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

# Simulation scale
NUM_SHIPMENTS         = 30
READINGS_PER_SHIPMENT = 5
START_TIMESTAMP       = datetime(2026, 5, 3, 6, 0, 0)
MAX_DEPARTURE_OFFSET_HOURS = 48   # Shipments depart within a 2-day window

# Temp-regulated categories — these get a Temperature sensor
TEMP_REGULATED_CATEGORIES = [
    "Deep Freeze",
    "Frozen",
    "Chill/Refrigerated",
    "Pharma",
    "Cool-Chain"
]

# Non-temp categories — GPS and RFID only, no Temperature sensor
NON_TEMP_CATEGORIES = [
    "Dry Goods",
    "Electronics",
    "Clothing",
    "Industrial"
]

GOODS_CATEGORIES = TEMP_REGULATED_CATEGORIES + NON_TEMP_CATEGORIES

# Temperature ranges per regulated category (Celsius) 
# Reference: https://www.cargo-wise.co.uk/blog/temperature-standards-for-cold-chain-logistics
TEMP_RANGES = {
    "Deep Freeze":        (-30.0, -28.0),
    "Frozen":             (-20.0, -16.0),
    "Chill/Refrigerated": (2.0,   4.0),
    "Pharma":             (2.0,   8.0),
    "Cool-Chain":         (12.0,  14.0)
}

# Real Philippine cities loaded from ph.csv;limited list
# Used for both origin/destination names and GPS coordinate interpolation
# 'ph.csv' Reference: https://simplemaps.com/data/ph-cities
cities_df = pd.read_csv("../data/ph.csv")
cities_df = cities_df[["city", "lat", "lng"]].dropna().drop_duplicates(subset="city") # guard against duplicate city names in ph.csv

CITIES      = cities_df["city"].tolist()
CITY_COORDS = dict(zip(cities_df["city"], zip(cities_df["lat"], cities_df["lng"])))

## 2. Shipment Registry
Generated first — all sensor scripts reference this. One row per shipment.
Assigns goods category, origin, destination, vehicle, and driver IDs.

### Output: `shipment_registry.csv`
Each row represents one shipment. This is the **master reference table** — every sensor reading links back to it via `rfid_tag`.

| Column | Format | Description |
|--------|--------|-------------|
| `rfid_tag` | `RFID-001` to `RFID-030` | Unique identifier for the shipment's RFID tag. This is the **foreign key** that connects all sensor tables together. |
| `goods_category` | e.g. `Frozen`, `Electronics` | The type of goods being shipped. Determines whether a Temperature sensor is attached. |
| `origin` | Philippine city name | The city where the shipment starts. Used for GPS coordinate interpolation. |
| `destination` | Philippine city name | The city where the shipment is headed. Always different from `origin`. |
| `package_count` | Integer 1–50 | Number of packages in the shipment. |
| `vehicle_id` | `VH-001` to `VH-030` | Unique ID of the delivery vehicle assigned to this shipment. |
| `driver_id` | `DR-001` to `DR-030` | Unique ID of the driver assigned to this shipment. |

In [2]:
registry_rows = []

for i in range(NUM_SHIPMENTS):
    rfid_tag = f"RFID-{i+1:03d}"
    origin   = random.choice(CITIES)
    dest     = random.choice([c for c in CITIES if c != origin])

    registry_rows.append({
        "rfid_tag":       rfid_tag,
        "goods_category": random.choice(GOODS_CATEGORIES),
        "origin":         origin,
        "destination":    dest,
        "package_count":  random.randint(1, 50),
        "vehicle_id":     f"VH-{i+1:03d}",
        "driver_id":      f"DR-{i+1:03d}",
        "departure_offset":  random.randint(0, MAX_DEPARTURE_OFFSET_HOURS),
    })

shipment_registry_df = pd.DataFrame(registry_rows)
print(f"Shipment Registry: {len(shipment_registry_df)} rows")
display(shipment_registry_df.head())

Shipment Registry: 30 rows


,rfid_tag,goods_category,origin,destination,package_count,vehicle_id,driver_id,departure_offset
0,RFID-001,Pharma,General Mariano Alvarez,San Andres,16,VH-001,DR-001,11
1,RFID-002,Dry Goods,San Isidro,Santamesa,43,VH-002,DR-002,24
2,RFID-003,Chill/Refrigerated,Pateros,Malabon,15,VH-003,DR-003,31
3,RFID-004,Deep Freeze,Bacoor,Valenzuela,17,VH-004,DR-004,45
4,RFID-005,Deep Freeze,Bayanan,Bagumbayan,3,VH-005,DR-005,30


## 3. GPS Sensor Readings
Every shipment gets GPS regardless of goods category.
Coordinates interpolate gradually from origin to destination using 
real city coordinates from `ph.csv`. Small random noise added so the 
path isn't a perfect straight line.

### Output: `gps_readings.csv`
Each row is one GPS ping from a vehicle. With 30 shipments × 5 readings each, this produces **150 rows**.

| Column | Format | Description |
|--------|--------|-------------|
| `reading_id` | `RDG-GPS-0001` ... | Globally unique ID for this GPS reading, sequentially numbered across all shipments. |
| `rfid_tag` | e.g. `RFID-007` | Links this GPS reading back to a specific shipment in the registry. |
| `device_id` | `GPS` + 3 random digits (e.g. `GPS412`) | Simulates the unique hardware ID of the GPS device installed on the vehicle. |
| `data_type` | `GPS` | Sensor type label. Used to distinguish rows in the combined `iot_data.csv`. |
| `data_value` | `lat,lng` (e.g. `14.599512,120.984219`) | The GPS coordinates at the time of the reading, formatted as `latitude,longitude`. |
| `timestamp` | `YYYY-MM-DD HH:MM:SS` | Time of the reading. Readings are spaced **2 hours apart**, starting from `START_TIMESTAMP`. |

### How GPS Coordinates Are Interpolated
The `fraction` variable moves from `0.0` (at origin) to `1.0` (at destination) across the 5 readings:

In [3]:
gps_rows = []

for _, shipment in shipment_registry_df.iterrows():
    shipment_start = START_TIMESTAMP + timedelta(hours=shipment["departure_offset"])
    origin_coords = CITY_COORDS[shipment["origin"]]
    dest_coords   = CITY_COORDS[shipment["destination"]]

    for j in range(READINGS_PER_SHIPMENT):
        # Gradually moves from origin to destination
        fraction = j / (READINGS_PER_SHIPMENT - 1)
        lat = round(origin_coords[0] + fraction * (dest_coords[0] - origin_coords[0]) + random.uniform(-0.05, 0.05), 6)
        lng = round(origin_coords[1] + fraction * (dest_coords[1] - origin_coords[1]) + random.uniform(-0.05, 0.05), 6)
        timestamp = shipment_start + timedelta(hours=j * 2)

        gps_rows.append({
            "reading_id": f"RDG-GPS-{len(gps_rows)+1:04d}",
            "rfid_tag":   shipment["rfid_tag"],
            "device_id":  f"GPS{random.randint(100,999)}",
            "data_type":  "GPS",
            "data_value": f"{lat},{lng}",
            "timestamp":  timestamp.strftime("%Y-%m-%d %H:%M:%S")
        })

gps_df = pd.DataFrame(gps_rows)

# Split data_value into typed lat/lng columns
gps_df["gps_lat"] = gps_df["data_value"].apply(lambda x: float(x.split(",")[0]))
gps_df["gps_lng"] = gps_df["data_value"].apply(lambda x: float(x.split(",")[1]))

print(f"GPS Readings: {len(gps_df)} rows")
display(gps_df.head())

GPS Readings: 150 rows


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp,gps_lat,gps_lng
0,RDG-GPS-0001,RFID-001,GPS985,GPS,"14.264923,120.967416",2026-05-03 17:00:00,14.264923,120.967416
1,RDG-GPS-0002,RFID-001,GPS963,GPS,"14.40099,120.962512",2026-05-03 19:00:00,14.400990,120.962512
2,RDG-GPS-0003,RFID-001,GPS174,GPS,"14.472531,120.968201",2026-05-03 21:00:00,14.472531,120.968201
3,RDG-GPS-0004,RFID-001,GPS477,GPS,"14.521417,120.977563",2026-05-03 23:00:00,14.521417,120.977563
4,RDG-GPS-0005,RFID-001,GPS168,GPS,"14.583952,120.974364",2026-05-04 01:00:00,14.583952,120.974364


## 4. RFID Sensor Readings
Every shipment gets RFID regardless of goods category.
Fires only at checkpoints — fewer readings than GPS (2–4 scans per shipment).
85% chance `VERIFIED`, 15% chance `FLAGGED` per scan.

### Output: `rfid_readings.csv`
Each row is one RFID checkpoint scan. With 2–4 scans per shipment, this produces roughly **90–120 rows** (randomized).

| Column | Format | Description |
|--------|--------|-------------|
| `reading_id` | `RDG-RFD-0001` ... | Globally unique ID for this RFID scan, sequentially numbered across all shipments. |
| `rfid_tag` | e.g. `RFID-007` | Links this scan back to a specific shipment in the registry. |
| `device_id` | `RFD` + 3 random digits (e.g. `RFD238`) | Simulates the unique hardware ID of the RFID scanner at the checkpoint. |
| `data_type` | `RFID` | Sensor type label. Used to distinguish rows in the combined `iot_data.csv`. |
| `data_value` | `VERIFIED` or `FLAGGED` | The result of the checkpoint scan (see table below). |
| `timestamp` | `YYYY-MM-DD HH:MM:SS` | Time of the scan. Gaps between scans are 2–6 hours apart — a fixed 4-hour base per scan index plus 0–2 hours of random variation. |

### `data_value` Status Codes
| Value | Probability | Meaning |
|-------|-------------|---------|
| `VERIFIED` | 85% | The RFID tag was successfully scanned and the shipment identity checks out. Package is on the correct route with no issues detected. |
| `FLAGGED` | 15% | The scan detected a problem — possible causes include tag damage, unexpected rerouting, tampered packaging, or shipment mismatch at the checkpoint. Requires manual follow-up. |

In [4]:
rfid_rows = []

for _, shipment in shipment_registry_df.iterrows():
    shipment_start = START_TIMESTAMP + timedelta(hours=shipment["departure_offset"])
    num_scans = random.randint(2, 4)

    for j in range(num_scans):
        timestamp = shipment_start + timedelta(hours=j * 4 + random.randint(0, 2))
        rfid_status = "VERIFIED" if random.random() > 0.15 else "FLAGGED"

        rfid_rows.append({
            "reading_id": f"RDG-RFD-{len(rfid_rows)+1:04d}",
            "rfid_tag":   shipment["rfid_tag"],
            "device_id":  f"RFD{random.randint(100,999)}",
            "data_type":  "RFID",
            "data_value": rfid_status,
            "timestamp":  timestamp.strftime("%Y-%m-%d %H:%M:%S")
        })

rfid_df = pd.DataFrame(rfid_rows)
print(f"RFID Readings: {len(rfid_df)} rows")
display(rfid_df.head())

RFID Readings: 84 rows


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp
0,RDG-RFD-0001,RFID-001,RFD169,RFID,VERIFIED,2026-05-03 18:00:00
1,RDG-RFD-0002,RFID-001,RFD713,RFID,VERIFIED,2026-05-03 21:00:00
2,RDG-RFD-0003,RFID-001,RFD970,RFID,VERIFIED,2026-05-04 02:00:00
3,RDG-RFD-0004,RFID-001,RFD549,RFID,VERIFIED,2026-05-04 05:00:00
4,RDG-RFD-0005,RFID-002,RFD315,RFID,VERIFIED,2026-05-04 08:00:00


## 5. Temperature Sensor Readings
Only temp-regulated categories receive a temperature sensor.
`Dry Goods`, `Electronics`, `Clothing`, and `Industrial` are skipped entirely.
10% chance of anomaly spike above the safe range per reading.

### Output: `temperature_readings.csv`
Each row is one temperature reading. Only shipments in a temp-regulated category produce rows here — the exact count depends on how many of the 30 shipments were randomly assigned a temp-regulated category.

| Column | Format | Description |
|--------|--------|-------------|
| `reading_id` | `RDG-TMP-0001` ... | Globally unique ID for this temperature reading, sequentially numbered. |
| `rfid_tag` | e.g. `RFID-003` | Links this reading back to a specific shipment in the registry. Only temp-regulated shipments appear here. |
| `device_id` | `TMP` + 3 random digits (e.g. `TMP891`) | Simulates the unique hardware ID of the temperature sensor attached to the shipment. |
| `data_type` | `Temperature` | Sensor type label. Used to distinguish rows in the combined `iot_data.csv`. |
| `data_value` | Decimal (°C), e.g. `3.7` or `-18.2` | The recorded temperature in Celsius at the time of the reading. |
| `timestamp` | `YYYY-MM-DD HH:MM:SS` | Time of the reading. Spaced **2 hours apart**, same interval as GPS readings. |

### Temperature Ranges & Anomaly Logic
Each category has a defined safe range. Per reading:
- **90% of the time** → temperature is within the safe range (`low` to `high`)
- **10% of the time** → an **anomaly spike** occurs: temperature rises to between `high` and `high + 3.0°C`, simulating a cold-chain breach

| Category | Safe Range | Anomaly Spike Range |
|----------|-----------|---------------------|
| `Deep Freeze` | −30°C to −28°C | −28°C to −25°C |
| `Frozen` | −20°C to −16°C | −16°C to −13°C |
| `Chill/Refrigerated` | 2°C to 4°C | 4°C to 7°C |
| `Pharma` | 2°C to 8°C | 8°C to 11°C |
| `Cool-Chain` | 12°C to 14°C | 14°C to 17°C |

In [5]:
temp_rows = []

for _, shipment in shipment_registry_df.iterrows():
    shipment_start = START_TIMESTAMP + timedelta(hours=shipment["departure_offset"])
    category = shipment["goods_category"]

    # Skip non-temp-regulated goods — no sensor attached
    if category not in TEMP_REGULATED_CATEGORIES:
        continue

    low, high = TEMP_RANGES[category]

    for j in range(READINGS_PER_SHIPMENT):
        timestamp = shipment_start + timedelta(hours=j * 2)

        # 10% chance of anomaly spike above safe range
        if random.random() < 0.10:
            temperature = round(random.uniform(high, high + 3.0), 1)
        else:
            temperature = round(random.uniform(low, high), 1)

        temp_rows.append({
            "reading_id": f"RDG-TMP-{len(temp_rows)+1:04d}",
            "rfid_tag":   shipment["rfid_tag"],
            "device_id":  f"TMP{random.randint(100,999)}",
            "data_type":  "Temperature",
            "data_value": temperature,
            "timestamp":  timestamp.strftime("%Y-%m-%d %H:%M:%S")
        })

temp_df = pd.DataFrame(temp_rows)

temp_df["data_value"] = temp_df["data_value"].astype(str)

print(f"Temperature Readings: {len(temp_df)} rows")
print(f"Skipped: {NON_TEMP_CATEGORIES}")
display(temp_df.head())

Temperature Readings: 95 rows
Skipped: ['Dry Goods', 'Electronics', 'Clothing', 'Industrial']


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp
0,RDG-TMP-0001,RFID-001,TMP706,Temperature,5.6,2026-05-03 17:00:00
1,RDG-TMP-0002,RFID-001,TMP191,Temperature,4.5,2026-05-03 19:00:00
2,RDG-TMP-0003,RFID-001,TMP182,Temperature,8.3,2026-05-03 21:00:00
3,RDG-TMP-0004,RFID-001,TMP458,Temperature,7.6,2026-05-03 23:00:00
4,RDG-TMP-0005,RFID-001,TMP133,Temperature,2.8,2026-05-04 01:00:00


## 6. Combine All Sensors
Stacks GPS, RFID, and Temperature into one unified iot_data.csv.
Sorted by timestamp so blockchain receives data chronologically.
Typed columns (gps_lat, gps_lng, temperature_c) are derived here
so the blockchain parser receives clean, schema-consistent fields
without needing to parse data_value itself.

### Output: `iot_data.csv`
This is the **master IoT feed** — every reading from all three sensor types in a single table, sorted by time. This is the file consumed by the blockchain processing loop.

| Column | Description |
|--------|-------------|
| `reading_id` | Unique ID per reading. Prefix indicates sensor type: `RDG-GPS-`, `RDG-RFD-`, or `RDG-TMP-`. |
| `rfid_tag` | The shipment this reading belongs to. Use this to join with `shipment_registry.csv`. |
| `device_id` | The hardware device that produced the reading (`GPS___`, `RFD___`, or `TMP___`). |
| `data_type` | One of `GPS`, `RFID`, or `Temperature` — identifies which sensor generated the row. |
| `data_value` | The sensor payload. Format depends on `data_type`: coordinates for GPS, status for RFID, decimal °C for Temperature. |
| `timestamp` | When the reading was taken. The table is sorted ascending by this column. |

### Row Count Breakdown
```
GPS rows         = 30 shipments × 5 readings          = 150 rows
RFID rows        = 30 shipments × 2–4 scans each       ≈ 90–120 rows
Temperature rows = (temp-regulated shipments only) × 5 ≈ variable
─────────────────────────────────────────────────────────────────────
Total            ≈ 300–350+ rows (exact count in Summary)
```

In [6]:
# NOTE for Week 6: when extracting numeric_value, use r'(-?\d+\.?\d*)' in your script
# NOT r'(\d+\.?\d*)', which is the default regex given on Camu's Code Template, as it drops the minus sign on negative temps


iot_data_df = pd.concat([gps_df, rfid_df, temp_df], ignore_index=True)
iot_data_df = iot_data_df.sort_values("timestamp").reset_index(drop=True)

# Derive typed columns so blockchain parser doesn't need to parse data_value itself
# gps_lat and gps_lng already exist from gps_df — concat carries them over
iot_data_df["temperature_c"] = iot_data_df.apply(
    lambda r: float(r["data_value"]) if r["data_type"] == "Temperature" else None, axis=1
)

print(f"Combined IoT Data: {len(iot_data_df)} total rows")
display(iot_data_df.head(10))

Combined IoT Data: 329 total rows


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp,gps_lat,gps_lng,temperature_c
0,RDG-TMP-0031,RFID-011,TMP104,Temperature,12.2,2026-05-03 07:00:00,NaN,NaN,12.2
1,RDG-GPS-0096,RFID-020,GPS291,GPS,"14.601956,120.989036",2026-05-03 07:00:00,14.601956,120.989036,NaN
2,RDG-GPS-0051,RFID-011,GPS728,GPS,"14.620032,120.962165",2026-05-03 07:00:00,14.620032,120.962165,NaN
3,RDG-RFD-0027,RFID-011,RFD702,RFID,VERIFIED,2026-05-03 08:00:00,NaN,NaN,NaN
4,RDG-GPS-0097,RFID-020,GPS920,GPS,"14.595757,120.981906",2026-05-03 09:00:00,14.595757,120.981906,NaN
5,RDG-GPS-0136,RFID-028,GPS356,GPS,"14.374937,121.038127",2026-05-03 09:00:00,14.374937,121.038127,NaN
6,RDG-RFD-0057,RFID-020,RFD166,RFID,VERIFIED,2026-05-03 09:00:00,NaN,NaN,NaN
7,RDG-TMP-0032,RFID-011,TMP505,Temperature,13.6,2026-05-03 09:00:00,NaN,NaN,13.6
8,RDG-GPS-0052,RFID-011,GPS262,GPS,"14.622406,120.970944",2026-05-03 09:00:00,14.622406,120.970944,NaN
9,RDG-RFD-0079,RFID-028,RFD922,RFID,FLAGGED,2026-05-03 10:00:00,NaN,NaN,NaN


## 7. Export
Saves all 5 CSVs. The `index=False` parameter ensures pandas does not write the DataFrame's row numbers as an extra column in the output files.

### Files Written
| File | Source DataFrame | Used By |
|------|-----------------|---------|
| `shipment_registry.csv` | `shipment_registry_df` | Reference table — join target for all sensor files |
| `gps_readings.csv` | `gps_df` | Location tracking and route analysis |
| `rfid_readings.csv` | `rfid_df` | Checkpoint audit trail and anomaly detection |
| `temperature_readings.csv` | `temp_df` | Cold-chain compliance monitoring |
| `iot_data.csv` | `iot_data_df` | **Primary input to the blockchain processing loop** |

In [ ]:
shipment_registry_df.drop(columns=["departure_offset"]).to_csv("../data/shipment_registry.csv", index=False)
gps_df.to_csv("../data/gps_readings.csv",                       index=False)
rfid_df.to_csv("../data/rfid_readings.csv",                     index=False)
temp_df.to_csv("../data/temperature_readings.csv",               index=False)
iot_data_df.to_csv("../data/iot_data.csv",                      index=False)

print("Files written:")
print("  → shipment_registry.csv")
print("  → gps_readings.csv")
print("  → rfid_readings.csv")
print("  → temperature_readings.csv")
print("  → iot_data.csv  - goes to the blockchain")

Files written:
  → shipment_registry.csv
  → gps_readings.csv
  → rfid_readings.csv
  → temperature_readings.csv
  → iot_data.csv  - goes to the blockchain


## 8. Summary
Breaks down all 3 sensors, shows how many shipments are temp-regulated vs not, and lists all 5 output files.

### What the Summary Reports
| Metric | How It's Calculated |
|--------|-------------------|
| **Total shipments** | Always equals `NUM_SHIPMENTS` (30) |
| **Temp-regulated** | Count of registry rows whose `goods_category` is in `TEMP_REGULATED_CATEGORIES` |
| **Non-temp** | `NUM_SHIPMENTS − temp_count` — these shipments have no temperature sensor |
| **GPS readings** | `len(gps_df)` — always `NUM_SHIPMENTS × READINGS_PER_SHIPMENT` = 150 |
| **RFID readings** | `len(rfid_df)` — varies (2–4 scans × 30 shipments) |
| **Temperature readings** | `len(temp_df)` — varies based on how many shipments were temp-regulated |
| **Total iot_data.csv rows** | Sum of all three sensors above |

In [8]:
temp_count     = shipment_registry_df[shipment_registry_df["goods_category"].isin(TEMP_REGULATED_CATEGORIES)].shape[0]
non_temp_count = NUM_SHIPMENTS - temp_count

print(f"""
Simulation Summary
------------------
Total shipments:              {NUM_SHIPMENTS}
  Temp-regulated:             {temp_count}
  Non-temp (no temp sensor):  {non_temp_count}

GPS readings:                 {len(gps_df)}
RFID readings:                {len(rfid_df)}
Temperature readings:         {len(temp_df)}
Total rows in iot_data.csv:   {len(iot_data_df)}

Files written:
  → shipment_registry.csv
  → gps_readings.csv
  → rfid_readings.csv
  → temperature_readings.csv
  → iot_data.csv
""")


Simulation Summary
------------------
Total shipments:              30
  Temp-regulated:             19
  Non-temp (no temp sensor):  11

GPS readings:                 150
RFID readings:                84
Temperature readings:         95
Total rows in iot_data.csv:   329

Files written:
  → shipment_registry.csv
  → gps_readings.csv
  → rfid_readings.csv
  → temperature_readings.csv
  → iot_data.csv

